# Notebook 03 - Training Deep Learning su Google Colab

## Scopo del notebook

Il **training del modello deep** (EfficientNet-B0 fine-tuned) richiede una
GPU e una buona quantita' di memoria. L'ambiente locale del progetto e'
CPU-only, quindi tutto il training va eseguito su:
- **Google Colab** (GPU T4 free tier) - consigliato.
- Oppure Kaggle Notebooks.

Questo notebook orchestrera':
1. Verifica della GPU disponibile.
2. Mount di Google Drive (per persistere i pesi).
3. Clone del repo + installazione delle dipendenze.
4. Copia/unzip del dataset.
5. Esecuzione di `scripts/train_deep.py`.
6. Plot delle training curves a partire dal log CSV.

## Setup richiesto (una tantum)

Prima di lanciare il notebook:
1. **Runtime -> Change runtime type -> T4 GPU**.
2. Carica `sarscov2-ctscan-dataset.zip` nella cartella `MyDrive/covid-ct/`
   del tuo Google Drive (manuale, da Drive web).
3. Sostituisci `REPO_URL` con l'URL del tuo fork del repository.


## 1. Verifica della GPU disponibile

In [ ]:
# `nvidia-smi`: comando NVIDIA che stampa info sulla GPU (modello,
# memoria, processi attivi). Se vediamo "Tesla T4" o simile, ottimo.
# Se la cella stampa un errore "command not found" significa che il
# runtime non ha GPU - cambialo da menu Runtime.
!nvidia-smi


## 2. Mount di Google Drive

I pesi del modello (`best_model.pth`) sono troppo grandi per committarli
nel repo. Li salviamo direttamente su Drive cosi' persistono dopo la
chiusura della sessione Colab (le sessioni Colab vengono cancellate
dopo qualche ora di inattivita').


In [ ]:
from google.colab import drive  # type: ignore[import-not-found]
# La prima volta apre un popup di autenticazione OAuth.
drive.mount('/content/drive')

# Tutti i nostri output finiranno in MyDrive/covid-ct/ del Drive
# dell'utente.
DRIVE_ROOT = '/content/drive/MyDrive/covid-ct'
import os; os.makedirs(DRIVE_ROOT, exist_ok=True)


## 3. Clone del repository

Colab da' un filesystem effimero ma una sessione "lunga" (max 12 ore).
Cloniamo il repo all'inizio e ci lavoriamo sopra. La logica gestisce
sia il primo clone sia gli aggiornamenti (`git pull`).


In [ ]:
import os
from IPython import get_ipython

# IMPORTANTE: sostituisci con l'URL del tuo repository GitHub.
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'
REPO_DIR = '/content/covid-ct'

# `get_ipython().system(...)` e' l'equivalente esplicito di `!cmd` nei
# notebook Jupyter/IPython. Lo usiamo (invece della scorciatoia `!`)
# perche' Pylance parsa `{REPO_URL} {REPO_DIR}` come due set literal
# adiacenti -> falso positivo "Espressione prevista". Usando f-string
# dentro una normale chiamata Python, il parser e' felice.
ipy = get_ipython()
if not os.path.exists(REPO_DIR):
    ipy.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    ipy.system(f'git -C {REPO_DIR} pull')

# `%cd` (cell magic) cambia la working directory per le celle successive
# (a differenza di `!cd` che cambierebbe SOLO per quella riga di shell).
ipy.run_line_magic('cd', REPO_DIR)


## 4. Installazione delle dipendenze

PyTorch + CUDA sono pre-installati su Colab. Aggiungiamo le librerie
specifiche del progetto: `timm` (backbone), `albumentations` (augmentation),
`scikit-image`, `scikit-learn`, `tqdm`, `pyyaml`, `grad-cam` (per
interpretabilita').


In [ ]:
# `--quiet` riduce l'output di pip (Colab tronca celle troppo lunghe).
# La cell magic `%pip` e' preferita alla scorciatoia shell perche'
# installa nell'ambiente Python del kernel del notebook (evita
# disallineamenti fra kernel attivo e python di sistema).
%pip install timm albumentations scikit-image scikit-learn tqdm pyyaml grad-cam --quiet


## 5a. Copia/unzip del dataset

Aspettiamo di trovare `sarscov2-ctscan-dataset.zip` su Drive (devi averlo
caricato manualmente prima della sessione). Lo unzippiamo nella working
directory del repo.


In [ ]:
import os
from IPython import get_ipython

DATASET_ZIP = f'{DRIVE_ROOT}/sarscov2-ctscan-dataset.zip'

# Skippiamo se la cartella esiste gia' (es. rerun del notebook).
if not os.path.exists('sarscov2-ctscan-dataset'):
    if os.path.exists(DATASET_ZIP):
        # `-q` quiet (no log riga per riga). `-d .` estrae nella cwd.
        # Usiamo `get_ipython().system(...)` invece di `!unzip ...` perche'
        # Pylance parsa `{DATASET_ZIP} -d` come `{set} -d` (sintassi
        # invalida) e genera un falso positivo "Espressione prevista".
        get_ipython().system(f'unzip -q {DATASET_ZIP} -d .')
    else:
        print('! Dataset not found. Upload sarscov2-ctscan-dataset.zip to Drive first.')


## 5b. Generazione degli split

Esegue `scripts/prepare_data.py` per creare i CSV `train/val/test`.
Da' un no-op se i file esistono gia' (idempotente).


In [ ]:
import os
from IPython import get_ipython

if not os.path.exists('data/processed/train.csv'):
    # `get_ipython().system(...)` invece di `!...`: Pylance parsa la
    # forma con `!` come Python regolare e si confonde su
    # `PYTHONPATH=. python` (interpreta `.python` come membro access ->
    # espressione invalida).
    get_ipython().system('PYTHONPATH=. python scripts/prepare_data.py')


## 6. Training

Lancia `scripts/train_deep.py` con i parametri canonici:
- `efficientnet_b0` come backbone (compromesso accuratezza/velocita').
- batch size 32 (ok su T4 con 16GB di VRAM).
- output salvato direttamente su Google Drive.

Su una T4 free-tier il training completo richiede ~30-60 min
(15 epoche x ~2 min). Le epoche reali dipendono dal config file.


In [ ]:
from IPython import get_ipython

OUTPUT_DIR = f'{DRIVE_ROOT}/models/deep'

# Comando di training assemblato come singola f-string e passato a
# `get_ipython().system(...)`. Evitiamo la sintassi `!cmd \ multilinea`
# perche' Pylance non gestisce il line-continuation bash dentro le cell
# magic (genera falsi positivi "Espressione prevista").
cmd = (
    'PYTHONPATH=. python scripts/train_deep.py'
    ' --arch efficientnet_b0'
    ' --batch-size 32'
    f' --output-dir {OUTPUT_DIR}'
)
get_ipython().system(cmd)


## 7. Plot delle training curves

Lo script di training salva un CSV (`training_log.csv`) con loss/accuracy/
AUC per ogni epoca. Lo leggiamo e plottiamo per ispezione visiva.

Cosa cercare:
- **Train loss in discesa, val loss anche** -> training sano.
- **Train loss scende ma val loss sale** -> overfitting (early stopping
  dovrebbe averlo gia' intercettato).
- **AUC che sale e si stabilizza** -> modello che converge.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f'{OUTPUT_DIR}/training_log.csv')

# 3 plot affiancati: Loss, Accuracy, AUC.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {1: '#e05c5c', 2: '#5c9be0'}   # rosso Fase 1, blu Fase 2

for phase, grp in log.groupby('phase'):
    c = colors[phase]
    label = f'Phase {phase}'
    # Train tratteggiata, val continua: convenzione visiva di molti paper.
    axes[0].plot(grp.index, grp['train_loss'], color=c, linestyle='--', label=f'{label} train')
    axes[0].plot(grp.index, grp['val_loss'],   color=c, linestyle='-',  label=f'{label} val')
    axes[1].plot(grp.index, grp['val_acc'],    color=c, label=label)
    axes[2].plot(grp.index, grp['val_auc'],    color=c, label=label)

for ax, title in zip(axes, ['Loss', 'Val Accuracy', 'Val AUC-ROC']):
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()

# Stampiamo anche il JSON con le metriche di test finali.
import json
with open(f'{OUTPUT_DIR}/test_results.json') as f:
    print(json.dumps(json.load(f), indent=2))
